In [1]:
import sys
sys.path.append("../src")

from utils import *

In [2]:
# import time
# class Oscillator:
#     """ A simple system clock that alternates between 0 and 1. """
#     def __init__(self, state = 0, frequency = 50):
#         self.frequency = frequency # cycles per second, HZ
#         self.state = state # state in 0 second
#         self.period = 1.0 / self.frequency # second
#     def __call__(self,):
#         time.sleep(self.period/2.0)
#         self.state = 1 - self.state
#         return self.state

# hardware simulator use something called Discrete Time Simulation.
# Rather than relying on wall-clock time, we simulate time using
# a loop of "ticks."


In [3]:
class Oscillator:
    """ A simple system clock that alternates between 0 and 1. """
    def __init__(self):
        self.state = 0

    def level(self):
        return self.state
        
    def tick(self):
        # Time moves forward one step!
        self.state = 1 - self.state
        return self.state

In [4]:
clock = Oscillator()
for tick in range(1, 21):
    # The Oscillator fires
    clk_signal = clock.tick()
    print(clk_signal)

1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0


In [5]:
clock.level()

0

In [6]:
# (Gemini:)I ignored the fact that one gate's output cascades 
# into the other gate's input within the exact same moment.
# class RS_FlipFlop:
#     def __init__(self):
#         self.nin = 2
#         self.nor_gate1 = NOR(2)
#         self.nor_gate2 = NOR(2)
        
#         # A flip-flop holds its state AND its inverted state
#         self.q = 0
#         self.q_bar = 1
        
#     def __call__(self, x):
#         assert len(x) == self.nin, "RS_FlipFlop takes exactly 2 inputs"
        
#         # x[0] is R (Reset)
#         # x[1] is S (Set)
#         r, s = x[0], x[1]

#         # Calculate the NEXT states simultaneously using the CURRENT memory
#         # Q = NOR(R, Q-bar)
#         next_q = self.nor_gate1([r, self.q_bar])
        
#         # Q-bar = NOR(S, Q)
#         next_q_bar = self.nor_gate2([s, self.q])

#         # Overwrite our internal memory with the new states
#         self.q = next_q
#         self.q_bar = next_q_bar
        
#         # Return both wires
#         return self.q, self.q_bar

In [7]:
class ResetSetFlipFlop:
    def __init__(self):
        self.nin = 2
        self.nor_gate1 = NOR()
        self.nor_gate2 = NOR()
        self.q = 0
        self.q_bar = 1 - self.q

    def getQ(self):
        return self.q

    def getQ_bar(self):
        return self.q_bar
    
    def __call__(self, x):
        assert len(x) == self.nin, "RS_FlipFlop takes exactly 2 inputs"
        r, s = x
        # 1. Q -> Q_bar -> Q
        # correct
        # self.q_bar = self.nor_gate2([self.q, s])
        # self.q = self.nor_gate1([self.q_bar, r])

        # 2. Q_bar -> Q -> Q_bar (- -> Q)
        # error
        # (PengChen:) because you evaluated Q first,
        # it didn't know that Q-bar was about to change!
        # Q evaluated using the "old" reality.
        # self.q = self.nor_gate1([self.q_bar, r])
        # self.q_bar = self.nor_gate2([self.q, s])

        # 3. iterate until stable
        while True:
            new_q = self.nor_gate1([r, self.q_bar])
            new_q_bar = self.nor_gate2([s, self.q])
            
            if new_q == self.q and new_q_bar == self.q_bar:
                break

            self.q = new_q
            self.q_bar = new_q_bar
        
        return self.q, self.q_bar

In [8]:
# 1. Power on the Flip-Flop
memory_cell = ResetSetFlipFlop()

def test_rs_latch(test_name, r, s):
    q, q_bar = memory_cell([r, s])
    print(f"R={r} S={s}  |  Q={q} (Q-bar={q_bar})  | {test_name}")

print("Testing RS Flip-Flop:")
print("-" * 50)

# 2. Sequential Testing (Time is moving forward!)

# State 1: Default
test_rs_latch("Initial State", 0, 0)

# State 2: Set the memory to 1
test_rs_latch("Set the bit to 1", 0, 1)

# State 3: Turn off the Set wire. It should REMEMBER the 1!
test_rs_latch("Turn inputs off (It REMEMBERS 1!)", 0, 0)

# State 4: Reset the memory to 0
test_rs_latch("Reset the bit to 0", 1, 0)

# State 5: Turn off the Reset wire. It should REMEMBER the 0!
test_rs_latch("Turn inputs off (It REMEMBERS 0!)", 0, 0)

# State 6: The "Forbidden" State Petzold talks about
test_rs_latch("The Forbidden State", 1, 1)

Testing RS Flip-Flop:
--------------------------------------------------
R=0 S=0  |  Q=0 (Q-bar=1)  | Initial State
R=0 S=1  |  Q=1 (Q-bar=0)  | Set the bit to 1
R=0 S=0  |  Q=1 (Q-bar=0)  | Turn inputs off (It REMEMBERS 1!)
R=1 S=0  |  Q=0 (Q-bar=1)  | Reset the bit to 0
R=0 S=0  |  Q=0 (Q-bar=1)  | Turn inputs off (It REMEMBERS 0!)
R=1 S=1  |  Q=0 (Q-bar=0)  | The Forbidden State


In [9]:
class LevelTriggeredDTypeFlipFlop:
    def __init__(self):
        self.nin = 2
        self.invert = Inverter()
        
        # Solder TWO physical AND gates to our board!
        self.and_gate1 = AND()
        self.and_gate2 = AND()
        
        self.rs_flipflop = ResetSetFlipFlop()

    def getQ(self):
        return self.rs_flipflop.getQ()

    def getQ_bar(self):
        return self.rs_flipflop.getQ_bar()
    
    def __call__(self, x):
        assert len(x) == self.nin, f"Expected {self.nin} inputs"
        data, clock = x[0], x[1]

        # Wire 1: Invert the Data
        not_data = self.invert([data])

        # Wire 2: The Reset pin (R) gets AND(NOT Data, Clock)
        r_wire = self.and_gate1([not_data, clock])

        # Wire 3: The Set pin (S) gets AND(Data, Clock)
        s_wire = self.and_gate2([data, clock])

        # Feed the R and S wires into our stabilized RS Flip-Flop
        q, q_bar = self.rs_flipflop([r_wire, s_wire])

        return q, q_bar

In [10]:
class LevelTriggeredDTypeFlipFlopWithClear:
    def __init__(self):
        self.nin = 3 # data, clk, clr
        self.invert_data = Inverter()
        self.invert_clr = Inverter() # added to protect the set pin
        
        # Solder TWO physical AND gates to our board!
        self.and_gate_r = AND(2)
        self.and_gate_s = AND(3) # upgraded to 3 inputs
        self.or_gate = OR()
        
        self.rs_flipflop = ResetSetFlipFlop()

    def getQ(self):
        return self.rs_flipflop.getQ()

    def getQ_bar(self):
        return self.rs_flipflop.getQ_bar()
    
    def __call__(self, x):
        assert len(x) in (self.nin, self.nin - 1), f"Expected {self.nin} or {self.nin - 1} inputs"
        if len(x) == self.nin:
            data, clock, clr = x
        else:
            data, clock = x
            clr = 0
            
        not_data = self.invert_data([data])
        not_clr = self.invert_clr([clr])
            
        # Wire 1: The Reset pin (R) gets OR(Clr, AND(NOT Data, Clock))
        and_r = self.and_gate_r([not_data, clock])
        r_wire = self.or_gate([clr, and_r])

        # Wire 2: The Set pin (S) gets AND(Data, Clock, NOT Clr)
        s_wire = self.and_gate_s([data, clock, not_clr])

        # Feed the R and S wires into our stabilized RS Flip-Flop
        q, q_bar = self.rs_flipflop([r_wire, s_wire])

        return q, q_bar

In [11]:
# Power on the D-Latch
d_latch = LevelTriggeredDTypeFlipFlopWithClear()

def test_latch(step, data, clock, note):
    q, q_bar = d_latch([data, clock])
    print(f"Step {step} | CLOCK={clock}  DATA={data} | Q={q} (Q-bar={q_bar}) | {note}")

print("Testing Level-Triggered D-Latch:")
print("-" * 80)

# 1. Start with everything off
test_latch(1, data=0, clock=0, note="Initial State")

# 2. Try to change data while the clock is 0 (The door is closed)
test_latch(2, data=1, clock=0, note="Data goes to 1, but Clock is 0. (IGNORED!)")

# 3. Turn the clock on (Open the door)
test_latch(3, data=1, clock=1, note="Clock goes to 1! Latch opens, Q mimics Data.")

# 4. Change data while clock is on
test_latch(4, data=0, clock=1, note="Data goes to 0 while Clock is 1. (Q follows!)")
test_latch(5, data=1, clock=1, note="Data goes to 1 while Clock is 1. (Q follows!)")

# 5. Turn the clock off (Slam the door shut!)
test_latch(6, data=1, clock=0, note="Clock goes to 0! Latch closes, REMEMBERING the 1.")

# 6. Try to change data again while the clock is 0
test_latch(7, data=0, clock=0, note="Data goes to 0, but Clock is 0. (IGNORED! Holds the 1)")

Testing Level-Triggered D-Latch:
--------------------------------------------------------------------------------
Step 1 | CLOCK=0  DATA=0 | Q=0 (Q-bar=1) | Initial State
Step 2 | CLOCK=0  DATA=1 | Q=0 (Q-bar=1) | Data goes to 1, but Clock is 0. (IGNORED!)
Step 3 | CLOCK=1  DATA=1 | Q=1 (Q-bar=0) | Clock goes to 1! Latch opens, Q mimics Data.
Step 4 | CLOCK=1  DATA=0 | Q=0 (Q-bar=1) | Data goes to 0 while Clock is 1. (Q follows!)
Step 5 | CLOCK=1  DATA=1 | Q=1 (Q-bar=0) | Data goes to 1 while Clock is 1. (Q follows!)
Step 6 | CLOCK=0  DATA=1 | Q=1 (Q-bar=0) | Clock goes to 0! Latch closes, REMEMBERING the 1.
Step 7 | CLOCK=0  DATA=0 | Q=1 (Q-bar=0) | Data goes to 0, but Clock is 0. (IGNORED! Holds the 1)


In [12]:
# Power on our new Clear-enabled Latch!
d_latch_with_clear = LevelTriggeredDTypeFlipFlopWithClear()

def test_latch_clear(step, data, clock, clr, note):
    q, q_bar = d_latch_with_clear([data, clock, clr])
    print(f"Step {step} | CLR={clr}  CLK={clock}  DATA={data} | Q={q} | {note}")

print("Testing Level-Triggered D-Latch with Asynchronous Clear:")
print("-" * 85)

# 1. Normal Operation
test_latch_clear(1, data=1, clock=1, clr=0, note="Normal Write: Saves a 1.")
test_latch_clear(2, data=0, clock=0, clr=0, note="Clock is 0: Door closed, memory holds the 1 safely.")

# 2. The Asynchronous Clear (Clock is still 0!)
test_latch_clear(3, data=0, clock=0, clr=1, note="CLEAR = 1! Even though Clock is 0, Q instantly resets to 0.")

# 3. Release the clear
test_latch_clear(4, data=0, clock=0, clr=0, note="CLEAR = 0. Memory stays at 0.")

# 4. The Override Test (Testing our Forbidden State fix)
test_latch_clear(5, data=1, clock=1, clr=1, note="CLEAR=1 while Data=1 and Clk=1. Clear OVERRIDES the write! Q stays 0.")

Testing Level-Triggered D-Latch with Asynchronous Clear:
-------------------------------------------------------------------------------------
Step 1 | CLR=0  CLK=1  DATA=1 | Q=1 | Normal Write: Saves a 1.
Step 2 | CLR=0  CLK=0  DATA=0 | Q=1 | Clock is 0: Door closed, memory holds the 1 safely.
Step 3 | CLR=1  CLK=0  DATA=0 | Q=0 | CLEAR = 1! Even though Clock is 0, Q instantly resets to 0.
Step 4 | CLR=0  CLK=0  DATA=0 | Q=0 | CLEAR = 0. Memory stays at 0.
Step 5 | CLR=1  CLK=1  DATA=1 | Q=0 | CLEAR=1 while Data=1 and Clk=1. Clear OVERRIDES the write! Q stays 0.


In [13]:
class NBitLevelTriggeredDTypeFlipFlopWithClear:
    def __init__(self, nbits):
        self.nbits = nbits
        # Solder 'n' D-Latches onto the board side-by-side
        self.level_d_latchs = [LevelTriggeredDTypeFlipFlopWithClear() for _ in range(self.nbits)]

    # We separate the data bus (a list) from the clock (a single bit)
    def __call__(self, datas, clk):
        assert len(datas) == self.nbits, f"Data bus must be {self.nbits} bits long"
        
        qs = []
        
        # Parallel processing! No rippling required.
        for idx in range(self.nbits):
            # Fetch the latch for this specific bit, and pass its data and the shared clock
            q, q_bar = self.level_d_latchs[idx]([datas[idx], clk])
            qs.append(q)
            
        return qs

In [14]:
# 1. Power on our 8-Bit Register!
register8 = NBitLevelTriggeredDTypeFlipFlopWithClear(8)

def test_register(step, num, clk, note):
    # Convert human number to hardware wires
    data_bits = int_to_8bit_list(num)
    
    # Run the hardware simulation
    q_bits = register8(data_bits, clk)
    
    # Read the memory back into a human number
    stored_num = bit_list_to_int(q_bits)
    
    print(f"Step {step} | CLK={clk} | Input Data: {num:3} | REGISTER HOLDS: {stored_num:3} | {note}")

print("Testing 8-Bit Register (n-Bit D-Latch):")
print("-" * 85)

# Step 1: Clock is OFF. Try to save the number 42.
test_register(1, 42, clk=0, note="Clock is 0. Data bounces off! (Register holds 0)")

# Step 2: Clock goes ON! The data (42) flows into the memory.
test_register(2, 42, clk=1, note="Clock is 1. Register successfully saves 42.")

# Step 3: Clock goes OFF. We lock the memory.
test_register(3, 42, clk=0, note="Clock is 0. Memory is locked.")

# Step 4: Try to save the number 255 while the clock is OFF.
test_register(4, 255, clk=0, note="Attempted to write 255. Clock is 0. Latch ignores it and REMEMBERS 42!")

# Step 5: Turn the Clock ON. The new data (255) overwrites the old memory.
test_register(5, 255, clk=1, note="Clock is 1. Register overwrites old memory and saves 255.")

Testing 8-Bit Register (n-Bit D-Latch):
-------------------------------------------------------------------------------------
Step 1 | CLK=0 | Input Data:  42 | REGISTER HOLDS:   0 | Clock is 0. Data bounces off! (Register holds 0)
Step 2 | CLK=1 | Input Data:  42 | REGISTER HOLDS:  42 | Clock is 1. Register successfully saves 42.
Step 3 | CLK=0 | Input Data:  42 | REGISTER HOLDS:  42 | Clock is 0. Memory is locked.
Step 4 | CLK=0 | Input Data: 255 | REGISTER HOLDS:  42 | Attempted to write 255. Clock is 0. Latch ignores it and REMEMBERS 42!
Step 5 | CLK=1 | Input Data: 255 | REGISTER HOLDS:  -1 | Clock is 1. Register overwrites old memory and saves 255.


In [15]:
# A circuit that open like a door (Level-Triggered) is 
# called a Latch.
# A circuit that only triggers on the clock's
# transition (Edge-Triggered) is called
# a Flip-Flop.
class EdgeTriggeredDTypeFlipFlop:
    def __init__(self):
        self.nin = 2

        self.level_d_latchs_stage1 = LevelTriggeredDTypeFlipFlop()
        self.level_d_latchs_stage2 = LevelTriggeredDTypeFlipFlop()
        self.invert1 = Inverter()
        # self.q, self.q_bar = 0, 1

    def getQ(self):
        return self.level_d_latchs_stage2.getQ()

    def getQ_bar(self):
        return self.level_d_latchs_stage2.getQ_bar()
        
    def __call__(self, x):
        assert len(x) == self.nin, f"Expected {self.nin} inputs"
        data, clock = x[0], x[1]

        # Wire 1: Invert the Clk
        not_clk = self.invert1([clock])

        # Wire 2: get Q and Q_bar from stage1
        q1, q_bar1 = self.level_d_latchs_stage1([data, not_clk])

        # Wire 3: get Q and Q_bar from stage2
        q2, q_bar2 = self.level_d_latchs_stage2([q1, clock])
        return q2, q_bar2

In [16]:
class ResetSetFlipFlop4Input:
    def __init__(self):
        # Edge-Triggered D-Type Flip-Flop with Clear and Preset (page 238)
        self.nin = 4 # r0, r1, s0, s1
        self.rs_flipflop = ResetSetFlipFlop()
        self.or_gate1 = OR()
        self.or_gate2 = OR()

    def getQ(self):
        return self.rs_flipflop.getQ()

    def getQ_bar(self):
        return self.rs_flipflop.getQ_bar()
    
    def __call__(self, x):
        assert len(x) == self.nin, f"RS_FlipFlop takes exactly {self.nin} inputs"
        r0, r1, s0, s1 = x
        r = self.or_gate1([r0, r1])
        s = self.or_gate2([s0, s1])

        q, q_bar = self.rs_flipflop([r, s])
        return q, q_bar

In [17]:
class EdgeTriggeredDTypeFlipFlopWithPresetAndClear:
    def __init__(self):
        self.nin = 4
        self.rs_flipflop1 = ResetSetFlipFlop4Input()
        self.rs_flipflop2 = ResetSetFlipFlop4Input()
        self.rs_flipflop3 = ResetSetFlipFlop4Input()
        self.invert1 = Inverter()
        
    def getQ(self):
        return self.rs_flipflop3.getQ()

    def getQ_bar(self):
        return self.rs_flipflop3.getQ_bar()
        
    def __call__(self, x):
        assert len(x) in (self.nin, self.nin-2), f"Expected {self.nin} inputs"
        if len(x) == self.nin:
            data, clock, preset, clear = x
        else:
            data, clock = x
            preset, clear = 0, 0

        not_clk = self.invert1([clock])
        while True:
            # 1. Take a snapshot of the current reality
            old_q1, old_q_bar1 = self.rs_flipflop1.getQ(), self.rs_flipflop1.getQ_bar()
            old_q2, old_q_bar2 = self.rs_flipflop2.getQ(), self.rs_flipflop2.getQ_bar()
            old_q3, old_q_bar3 = self.rs_flipflop3.getQ(), self.rs_flipflop3.getQ_bar()

            # 2. Let the electrons flow through all the wires
            self.rs_flipflop1([self.rs_flipflop2.getQ_bar(), not_clk, data, preset])
            self.rs_flipflop2([clear, self.rs_flipflop1.getQ_bar(), preset, not_clk])
            q3, q_bar3 = self.rs_flipflop3([clear, self.rs_flipflop2.getQ_bar(), self.rs_flipflop1.getQ(), preset])    

            # 3. Did the electrons change anything? If no, the circuit is stable!
            if (old_q1 == self.rs_flipflop1.getQ() and old_q_bar1 == self.rs_flipflop1.getQ_bar() and 
                old_q2 == self.rs_flipflop2.getQ() and old_q_bar2 == self.rs_flipflop2.getQ_bar() and 
                old_q3 == self.rs_flipflop3.getQ() and old_q_bar3 == self.rs_flipflop3.getQ_bar()):
                break;
        
        return q3, q_bar3

In [18]:
# Power on the V2 Flip-Flop!
d_ff_v2 = EdgeTriggeredDTypeFlipFlopWithPresetAndClear()

def test_ff_v2(step, data, clock, preset, clear, note):
    q, q_bar = d_ff_v2([data, clock, preset, clear])
    print(f"Step {step} | PRE={preset} CLR={clear} CLK={clock} DATA={data} | Q={q} | {note}")

print("Testing Edge-Triggered D-Flip-Flop V2 (With Preset & Clear):")
print("-" * 90)

# 1. Normal Edge-Triggered Operation
test_ff_v2(1, data=1, clock=0, preset=0, clear=0, note="Clock 0. Data is 1. Q stays 0.")
test_ff_v2(2, data=1, clock=1, preset=0, clear=0, note="RISING EDGE! Q becomes 1.")
test_ff_v2(3, data=0, clock=1, preset=0, clear=0, note="Clock stays 1. Data drops to 0. Q IGNORES it (Q=1).")
test_ff_v2(4, data=0, clock=0, preset=0, clear=0, note="FALLING EDGE. Q stays 1.")

# 2. Testing Asynchronous Clear
test_ff_v2(5, data=0, clock=0, preset=0, clear=1, note="CLEAR = 1! Instantly forces Q to 0.")
test_ff_v2(6, data=0, clock=0, preset=0, clear=0, note="Clear released. Q stays 0.")

# 3. Testing Asynchronous Preset
test_ff_v2(7, data=0, clock=0, preset=1, clear=0, note="PRESET = 1! Instantly forces Q to 1.")
test_ff_v2(8, data=0, clock=0, preset=0, clear=0, note="Preset released. Q stays 1.")

# 4. The Override Test
test_ff_v2(9, data=1, clock=1, preset=0, clear=1, note="RISING EDGE tries to save Data=1, but CLEAR=1 overpowers it! (Q=0)")

Testing Edge-Triggered D-Flip-Flop V2 (With Preset & Clear):
------------------------------------------------------------------------------------------
Step 1 | PRE=0 CLR=0 CLK=0 DATA=1 | Q=0 | Clock 0. Data is 1. Q stays 0.
Step 2 | PRE=0 CLR=0 CLK=1 DATA=1 | Q=1 | RISING EDGE! Q becomes 1.
Step 3 | PRE=0 CLR=0 CLK=1 DATA=0 | Q=1 | Clock stays 1. Data drops to 0. Q IGNORES it (Q=1).
Step 4 | PRE=0 CLR=0 CLK=0 DATA=0 | Q=1 | FALLING EDGE. Q stays 1.
Step 5 | PRE=0 CLR=1 CLK=0 DATA=0 | Q=0 | CLEAR = 1! Instantly forces Q to 0.
Step 6 | PRE=0 CLR=0 CLK=0 DATA=0 | Q=0 | Clear released. Q stays 0.
Step 7 | PRE=1 CLR=0 CLK=0 DATA=0 | Q=1 | PRESET = 1! Instantly forces Q to 1.
Step 8 | PRE=0 CLR=0 CLK=0 DATA=0 | Q=1 | Preset released. Q stays 1.
Step 9 | PRE=0 CLR=1 CLK=1 DATA=1 | Q=0 | RISING EDGE tries to save Data=1, but CLEAR=1 overpowers it! (Q=0)


In [19]:
# Power on the Master-Slave D Flip-Flop!
edge_d_flipflop = EdgeTriggeredDTypeFlipFlopWithPresetAndClear()

def test_edge(step, data, clock, note):
    q, q_bar = edge_d_flipflop([data, clock])
    print(f"Step {step} | CLK={clock}  DATA={data} | Q={q} | {note}")

print("Testing Edge-Triggered D Flip-Flop (Master-Slave):")
print("-" * 85)

# 1. Default state
test_edge(1, data=0, clock=0, note="Initial State. Master is tracking 0, Slave is locked.")

# 2. Change data while clock is 0
test_edge(2, data=1, clock=0, note="Data=1. Master tracks the 1, but Slave holds Q=0.")

# 3. THE POSITIVE EDGE (Clock goes from 0 to 1)
test_edge(3, data=1, clock=1, note="RISING EDGE! Master locks the 1, Slave opens to output Q=1.")

# 4. The Danger Zone: Change data while Clock is STILL 1
test_edge(4, data=0, clock=1, note="Data=0 while CLK is 1. Master is LOCKED, so Q completely IGNORES the change! (Q=1)")

# 5. The Negative Edge (Clock goes from 1 to 0)
test_edge(5, data=0, clock=0, note="FALLING EDGE. Master opens to track 0. Slave closes, holding Q=1.")

# 6. Another Positive Edge
test_edge(6, data=0, clock=1, note="RISING EDGE! Master locks the 0, Slave opens to output Q=0.")

Testing Edge-Triggered D Flip-Flop (Master-Slave):
-------------------------------------------------------------------------------------
Step 1 | CLK=0  DATA=0 | Q=0 | Initial State. Master is tracking 0, Slave is locked.
Step 2 | CLK=0  DATA=1 | Q=0 | Data=1. Master tracks the 1, but Slave holds Q=0.
Step 3 | CLK=1  DATA=1 | Q=1 | RISING EDGE! Master locks the 1, Slave opens to output Q=1.
Step 4 | CLK=1  DATA=0 | Q=1 | Data=0 while CLK is 1. Master is LOCKED, so Q completely IGNORES the change! (Q=1)
Step 5 | CLK=0  DATA=0 | Q=1 | FALLING EDGE. Master opens to track 0. Slave closes, holding Q=1.
Step 6 | CLK=1  DATA=0 | Q=0 | RISING EDGE! Master locks the 0, Slave opens to output Q=0.


In [20]:
class NBitsEdgeTriggeredDTypeFlipFlopWithPresetAndClear:
    def __init__(self, nbits):
        self.nbits = nbits
        
        # Solder 'n' Edge-Triggered D-Flip-Flops onto the board side-by-side
        self.flip_flops = [EdgeTriggeredDTypeFlipFlopWithPresetAndClear() for _ in range(self.nbits)]

    def __call__(self, datas, clk):
        assert len(datas) == self.nbits, f"Data bus must be {self.nbits} bits long"
        
        qs = []
        
        # Parallel processing! All bits lock in at the exact same time.
        for idx in range(self.nbits):
            q, q_bar = self.flip_flops[idx]([datas[idx], clk])
            qs.append(q)
            
        return qs

In [21]:
# Power on our 8-Bit Edge-Triggered Register!
cpu_register = NBitsEdgeTriggeredDTypeFlipFlopWithPresetAndClear(8)

def test_edge_register(step, num, clk, note):
    # Convert human number to 8-bit hardware wire states
    data_bits = int_to_8bit_list(num)
    
    # Run the hardware forward pass
    q_bits = cpu_register(data_bits, clk)
    
    # Read the memory back into a human number
    stored_num = bit_list_to_int(q_bits)
    
    print(f"Step {step} | CLK={clk} | Input Data: {num:3} | REGISTER HOLDS: {stored_num:3}")
    print(f"       -> {note}\n")

print("Testing 8-Bit Edge-Triggered Register:")
print("-" * 80)

# Step 1: Clock is 0. Data is 100.
test_edge_register(1, 100, clk=0, note="Clock is 0. Register is closed. (Holds 0)")

# Step 2: RISING EDGE! Clock goes to 1. Data is 100.
test_edge_register(2, 100, clk=1, note="RISING EDGE! Register locks in the 100.")

# Step 3: THE DANGER ZONE. Clock stays at 1. Data changes to 255.
test_edge_register(3, 255, clk=1, note="Clock is STILL 1. Data changed to 255, but Register IGNORES it! (Still holds 100)")

# Step 4: FALLING EDGE. Clock goes to 0. Data is 255.
test_edge_register(4, 255, clk=0, note="FALLING EDGE. Register is closed. (Still holds 100)")

# Step 5: RISING EDGE! Clock goes to 1. Data is 255.
test_edge_register(5, 255, clk=1, note="RISING EDGE! Register finally locks in the new 255.")

Testing 8-Bit Edge-Triggered Register:
--------------------------------------------------------------------------------
Step 1 | CLK=0 | Input Data: 100 | REGISTER HOLDS:   0
       -> Clock is 0. Register is closed. (Holds 0)

Step 2 | CLK=1 | Input Data: 100 | REGISTER HOLDS: 100
       -> RISING EDGE! Register locks in the 100.

Step 3 | CLK=1 | Input Data: 255 | REGISTER HOLDS: 100
       -> Clock is STILL 1. Data changed to 255, but Register IGNORES it! (Still holds 100)

Step 4 | CLK=0 | Input Data: 255 | REGISTER HOLDS: 100
       -> FALLING EDGE. Register is closed. (Still holds 100)

Step 5 | CLK=1 | Input Data: 255 | REGISTER HOLDS:  -1
       -> RISING EDGE! Register finally locks in the new 255.



In [22]:
class NBitsAccumulator:
    def __init__(self, nbits):
        self.nbits = nbits
        self.adder = NBitAdderWithOverflow(self.nbits)
        self.register = NBitsEdgeTriggeredDTypeFlipFlopWithPresetAndClear(self.nbits)
        # At boot-up, the memory holds all zeros
        self.current_memory = [0 for _ in range(self.nbits)]

    def __call__(self, inputs_sequence):
        # We process a list of binary numbers, simulating time passing
        for data_in in inputs_sequence:
            assert len(data_in) == self.nbits , f"Data bus must be {self.nbits} bits long"
            # 1. The adder adds the incoming number to the CURRENT memory
            overflow, carry, sum_bits = self.adder(data_in, self.current_memory)

            # 2. Manually toggle the clock (Press the "Add" button)
            self.register(sum_bits, 0)
            # RISING EDGE: Button goes down (Clk=1). The register locks in the new sum
            self.current_memory = self.register(sum_bits, 1)
            # Button goes back up (Clk = 0)
            self.register(sum_bits, 0)
        # Return whatever the register is holding at the very end
        return self.current_memory

    def clear(self):
        """
        Clears the accumulator using pure hardward math.
        Adds the Two's complement of the current memory to itself.
        X + NOT(X) + 1 = 0
        """
        my_inverter = Inverter()

        inverted_memory = [my_inverter([bit]) for bit in self.current_memory]
        overflow, carry, sum_bits = self.adder(inverted_memory, self.current_memory, last_carry_in=1)

        # 2. Manually toggle the clock (Press the "Add" button)
        self.register(sum_bits, 0)
        # RISING EDGE: Button goes down (Clk=1). The register locks in the new sum
        self.current_memory = self.register(sum_bits, 1)
        # Button goes back up (Clk = 0)
        self.register(sum_bits, 0)        

In [23]:
# Power on our 8-Bit Accumulator!
accumulator = NBitsAccumulator(8)

def test_accumulator(sequence_of_numbers):
    print(f"Feeding sequence into Accumulator: {sequence_of_numbers}")
    print("-" * 50)
    
    # 1. Convert the human numbers into a list of 8-bit wire arrays
    hardware_inputs = [int_to_8bit_list(num) for num in sequence_of_numbers]
        
    # 2. Run the hardware simulation
    accumulator.clear()
    final_bits = accumulator(hardware_inputs)
    
    # 3. Read the final state of the register
    final_total = bit_list_to_int(final_bits)
    
    print(f"Final Register Bits: {final_bits}")
    print(f"DECIMAL TOTAL:       {final_total}")
    print(f"Expected Math:       {sum(sequence_of_numbers)}")
    print("\n")

# Test 1: Standard Addition
test_accumulator([10, 20, 30])

# Test 2: Adding a negative number (Two's Complement in action!)
# 50 + 25 - 15 = 60
test_accumulator([50, 25, -15])

# Test 3: The Big Loop
test_accumulator([1, 2, 3, 4, 5, 6, 7, 8, 9, 10]) # Should be 55

Feeding sequence into Accumulator: [10, 20, 30]
--------------------------------------------------
Final Register Bits: [0, 0, 1, 1, 1, 1, 0, 0]
DECIMAL TOTAL:       60
Expected Math:       60


Feeding sequence into Accumulator: [50, 25, -15]
--------------------------------------------------
Final Register Bits: [0, 0, 1, 1, 1, 1, 0, 0]
DECIMAL TOTAL:       60
Expected Math:       60


Feeding sequence into Accumulator: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
--------------------------------------------------
Final Register Bits: [0, 0, 1, 1, 0, 1, 1, 1]
DECIMAL TOTAL:       55
Expected Math:       55




In [24]:
class FrequencyDivider:
    def __init__(self):
        self.nin = 2
        # init self.state = 0
        self.oscillator = Oscillator() 
        self.flipflop = EdgeTriggeredDTypeFlipFlop()

        
    def __call__(self):
        clk = self.oscillator.level()
        self.oscillator.tick()
        data = self.flipflop.getQ_bar()

        q, q_bar = self.flipflop([data, clk])
        return q, q_bar

In [25]:
# Power on the Frequency Divider
divider = FrequencyDivider()

print("Tick | CLK |  Q   (Q-bar)")
print("-" * 27)

# Let time run for 10 "ticks" (5 full clock cycles)
for tick in range(1, 11):
    # Tap the clock wire so we can print it
    current_clk = divider.oscillator.level()
    
    # Run the hardware simulation
    q, q_bar = divider()
    
    # Print the states side-by-side to see the waves!
    print(f" {tick:2}  |  {current_clk}  |  {q}      {q_bar}")

Tick | CLK |  Q   (Q-bar)
---------------------------
  1  |  0  |  0      1
  2  |  1  |  1      0
  3  |  0  |  1      0
  4  |  1  |  0      1
  5  |  0  |  0      1
  6  |  1  |  1      0
  7  |  0  |  1      0
  8  |  1  |  0      1
  9  |  0  |  0      1
 10  |  1  |  1      0


In [26]:
class NBitsFrequencyDivider:
    def __init__(self, nbits):
        self.nbits = nbits
        self.flipflops = [EdgeTriggeredDTypeFlipFlopWithPresetAndClear() for _ in range(self.nbits)]

    def __call__(self, clk = 0):
        current_clk = clk
        qs = []
        for flipflop in self.flipflops:
            data = flipflop.getQ_bar()
            q, q_bar = flipflop([data, current_clk])
            qs.append(q)
            current_clk = q_bar
        # MSB is on the left
        return list(reversed(qs))

In [28]:
# 1. Power on the System Clock and the 4-Bit Counter
system_clock = Oscillator()
counter4 = NBitsFrequencyDivider(4)

print("Tick | !CLK | Q3 Q2 Q1 Q0 | Decimal")
print("-" * 36)

# 2. Run the simulation for 34 ticks to watch it count 0 to 15, and wrap around
for tick in range(1, 35):
    # Tap the clock wire to see its state
    clk_signal = system_clock.level()
    
    # Run the counter hardware using the current clock signal
    q_bits = counter4(clk_signal)
    
    # Read the binary list into a human decimal number
    decimal_val = bit_list_to_int(q_bits, signed=False)
    
    # Print the state
    print(f" {tick:2}  |  {1-clk_signal}  |  {q_bits[0]}  {q_bits[1]}  {q_bits[2]}  {q_bits[3]}  |   {decimal_val:2}")
    
    # Tick the clock forward for the next loop iteration
    system_clock.tick()

Tick | !CLK | Q3 Q2 Q1 Q0 | Decimal
------------------------------------
  1  |  1  |  0  0  0  0  |    0
  2  |  0  |  0  0  0  1  |    1
  3  |  1  |  0  0  0  1  |    1
  4  |  0  |  0  0  1  0  |    2
  5  |  1  |  0  0  1  0  |    2
  6  |  0  |  0  0  1  1  |    3
  7  |  1  |  0  0  1  1  |    3
  8  |  0  |  0  1  0  0  |    4
  9  |  1  |  0  1  0  0  |    4
 10  |  0  |  0  1  0  1  |    5
 11  |  1  |  0  1  0  1  |    5
 12  |  0  |  0  1  1  0  |    6
 13  |  1  |  0  1  1  0  |    6
 14  |  0  |  0  1  1  1  |    7
 15  |  1  |  0  1  1  1  |    7
 16  |  0  |  1  0  0  0  |    8
 17  |  1  |  1  0  0  0  |    8
 18  |  0  |  1  0  0  1  |    9
 19  |  1  |  1  0  0  1  |    9
 20  |  0  |  1  0  1  0  |   10
 21  |  1  |  1  0  1  0  |   10
 22  |  0  |  1  0  1  1  |   11
 23  |  1  |  1  0  1  1  |   11
 24  |  0  |  1  1  0  0  |   12
 25  |  1  |  1  1  0  0  |   12
 26  |  0  |  1  1  0  1  |   13
 27  |  1  |  1  1  0  1  |   13
 28  |  0  |  1  1  1  0  |   14
 29